In [38]:
# %pip install pypdf

In [39]:
import os
import numpy as np
import time
from pypdf import PdfReader
from common import client, ask

In [40]:
PDF = "오로라테크_사내규정집_v7.0.pdf"
# EMBED_MODEL = "gemini-embedding-002"
EMBED_MODEL = "gemini-embedding-2"

# 04예제는 저장될 문서가 문자열 리스트이고, 지금 예제는 pdf

# 1. pdf -> 조각내기
# size : 몇자씩 조각낼건지
# overlap : 몇자씩 겹치게 할건지
def load_chunks(path, size=500, overlap=100):
    chunks = []
    for page_no, page in enumerate(PdfReader(path).pages, start=1):
        # page.extract_text() : pdf 페이지 내용
        # .replace("\n", " ") : 엔터를 스페이스로 변경
        # .strip() : 시작 앞 공백과 마지막 글자 뒤 공백을 제거
        text = (page.extract_text() or "").replace("\n", " ").strip()
        # size 개수만큼 문자열을 가져오고, 다음 작업은 100개가 겹치게 시작점을 지정
        for start in range(0, len(text), size - overlap):
            # size 개수만큼 문자열을 가져옴
            piece = text[start: start+size]
            if len(piece) > 50: # 작은 조각()은 버림
                # 답변할 때 규정집 어디에 있는지 알리기 위해 페이지 번호도 함께
                chunks.append({"page" : page_no, "text" : piece})
    return chunks

In [41]:
chunks = load_chunks(PDF)

In [42]:
# 04예제와 다른점은 04 예제는 임베딩할 문자열이 몇개 안되서 통째로 임베딩을 함
# 이 예제는 문서를 불러와서 임베딩 하기 때문에 임베딩할 문자열이 많을 수 있음
# batch를 이용하여 임베딩할 문자열 개수를 조절하여 나누어서 임베딩을 함
def embed(texts, batch=50):
    vecs = []
    if EMBED_MODEL == "gemini-embedding-001":
        for i in range(0, len(texts), batch):
            r = client.models.embed_content(model=EMBED_MODEL, contents=texts)
            vecs += [e.values for e in r.embeddings]
            time.sleep(0.5)
    elif EMBED_MODEL == "gemini-embedding-2":
        count = 0
        for text in texts:
            r = client.models.embed_content(model=EMBED_MODEL, contents=text)
            vecs.append(r.embeddings[0].values)
            count+=1
            print(f"{count}/{len(texts)} 완료")
            time.sleep(0.5)
    return np.array(vecs)

In [43]:
VECS_FILE = "pdf_vecs.npy"
if os.path.exists(VECS_FILE):
    vecs = np.load(VECS_FILE)
else:
    vecs = embed([c.get("text","") for c in chunks])
    np.save(VECS_FILE, vecs)
print(f"조각 {len(chunks)}개 준비완료")

1/76 완료
2/76 완료
3/76 완료
4/76 완료
5/76 완료
6/76 완료
7/76 완료
8/76 완료
9/76 완료
10/76 완료
11/76 완료
12/76 완료
13/76 완료
14/76 완료
15/76 완료
16/76 완료
17/76 완료
18/76 완료
19/76 완료
20/76 완료
21/76 완료
22/76 완료
23/76 완료
24/76 완료
25/76 완료
26/76 완료
27/76 완료
28/76 완료
29/76 완료
30/76 완료
31/76 완료
32/76 완료
33/76 완료
34/76 완료
35/76 완료
36/76 완료
37/76 완료
38/76 완료
39/76 완료
40/76 완료
41/76 완료
42/76 완료
43/76 완료
44/76 완료
45/76 완료
46/76 완료
47/76 완료
48/76 완료
49/76 완료
50/76 완료
51/76 완료
52/76 완료
53/76 완료
54/76 완료
55/76 완료
56/76 완료
57/76 완료
58/76 완료
59/76 완료
60/76 완료
61/76 완료
62/76 완료
63/76 완료
64/76 완료
65/76 완료
66/76 완료
67/76 완료
68/76 완료
69/76 완료
70/76 완료
71/76 완료
72/76 완료
73/76 완료
74/76 완료
75/76 완료
76/76 완료
조각 76개 준비완료


In [46]:
def search(question, k=3):
    q = embed([question])[0]
    # 코사인 유사도를 계산. 저장된 문서 중 질문과 가장 가까운 문서를 찾기 위해
    sims = vecs @ q / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(q))
    # 유사도가 높은 순으로 정렬 후 k개를 선택
    top = np.argsort(sims)[::-1][:k]          # 2. 검색: 유사도 높은 순
    return [(chunks[i], round(float(sims[i]), 3)) for i in top]

In [47]:
def answer(question):
    found = search(question)
    context = "\n".join(f"[{i+1}] {t}" for i, (t, _) in enumerate(found))
    r = ask(f"""아래 문서에 있는 내용으로만 답하라. 문서에 없으면 '자료에 없습니다'라고 답한다.
    답 끝에 참고한 페이지를 (p.번호) 형식으로 적는다.
    
    {context}
    
    질문: {question}""", temperature=0)
    return r.text

In [ ]:
while True:
    q = input("질문 : ")
    if q == "그만":
        break
    print(answer(q), "\n")

질문 :  입사 3달됐는데 휴가 얼마나 있어?


1/1 완료
  (gemini-3.1-flash-lite 응답)
계속 근로 기간이 1년 미만인 직원은 1개월 개근 시 1일의 유급휴가가 부여되므로, 3개월 개근 시 총 3일의 유급휴가가 있습니다. (p.11) 

